# 02b — Currency Transactions

Procesa `currency_transactions.csv` para generar features agregadas por
usuario sobre actividad económica.

**Inputs:**
- `data/data_raw/currency_transactions.csv` (5.1M filas)
- `data/data_qc/sample_user_ids.parquet` (heredado de 02a)

**Outputs:**
- `data/data_qc/currency_transactions_filtered.parquet` (transacciones filtradas)
- `data/data_qc/features_currency.parquet` (features agregadas, 1 fila/usuario)

**Hallazgo clave:** Las monedas `pve_energy`, `experience`, `mana` y
`tower_event_points` son 100% transacciones del sistema (user_id nulo).
Solo `gold`, `gems` y `dark_steel` tienen datos a nivel de usuario.

**Principio de robustez:** este notebook carga `sample_user_ids.parquet`
como filtro maestro. Si cambian los filtros del 02a, basta re-ejecutar
ambos notebooks sin tocar código.

In [1]:
# [SETUP] Imports y dependencias
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gc
import time
from pathlib import Path
from datetime import datetime

plt.style.use('default')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

In [ ]:
# [SETUP] Paths, constantes, semilla y helpers
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase1_cleaning' else Path.cwd()
DATA_RAW = PROJECT_ROOT / 'data' / 'data_raw'
DATA_QC = PROJECT_ROOT / 'data' / 'data_qc'
REPORT_DIR = PROJECT_ROOT / 'informes' / 'fase1_cleaning' / 'currency_transactions'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

CSV_INPUT = DATA_RAW / 'currency_transactions.csv'
SAMPLE_IDS_PATH = DATA_QC / 'sample_user_ids_cutoff.parquet'
PARQUET_FILTERED = DATA_QC / 'currency_transactions_filtered.parquet'
PARQUET_FEATURES = DATA_QC / 'features_currency_cutoff.parquet'

# Monedas con datos reales a nivel de usuario (verificado en iteración 1)
# pve_energy, experience, mana, tower_event_points son 100% sistema (user_id nulo)
TOP_CURRENCIES = ['gold', 'gems', 'dark_steel']

# Concepts con datos reales a nivel de usuario (verificado en iteración 1)
TOP_CONCEPTS = [4, 2, 0, 10, 12, 46, 54]

# Helper: normalizar user_id de ObjectId a hex limpio (salvaguarda)
def clean_user_id(uid):
    uid = str(uid)
    if uid.startswith('ObjectId(') and uid.endswith(')'):
        return uid[9:-1].replace("'", "")
    return uid

# Helper: log de pasos
steps_log = []
NOTEBOOK_START = time.time()

def log_step(tag, message):
    ts = datetime.now().strftime('%H:%M:%S')
    entry = f"[{tag}] {ts} — {message}"
    steps_log.append(entry)
    print(entry)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CSV_INPUT existe: {CSV_INPUT.exists()}")
print(f"SAMPLE_IDS_PATH existe: {SAMPLE_IDS_PATH.exists()}")
print(f"\nTOP_CURRENCIES: {TOP_CURRENCIES}")
print(f"TOP_CONCEPTS: {TOP_CONCEPTS}")

In [ ]:
# [SETUP] Carga de sample_user_ids y REFERENCE_DATE
sample_ids_df = pd.read_parquet(SAMPLE_IDS_PATH)

# Normalizar user_id por si acaso viene con ObjectId wrapper
sample_ids_df['user_id'] = sample_ids_df['user_id'].apply(clean_user_id)
sample_user_ids = set(sample_ids_df['user_id'])
N_SAMPLE = len(sample_user_ids)

print(f"Usuarios en sample: {N_SAMPLE:,}")
print(f"Formato user_id (ejemplo): '{list(sample_user_ids)[0]}' (len={len(list(sample_user_ids)[0])})")

# Verificar que es hex limpio
sample_example = list(sample_user_ids)[0]
assert len(sample_example) == 24 and not sample_example.startswith('ObjectId'), \
    f"ERROR: user_id no es hex limpio: '{sample_example}'"

# REFERENCE_DATE heredado de users_clean
users_clean = pd.read_parquet(DATA_QC / 'users_clean_cutoff.parquet', columns=['last_login_dt'])
REFERENCE_DATE = users_clean['last_login_dt'].max()
CUTOFF_DATE = REFERENCE_DATE - pd.Timedelta(days=120)
del users_clean
gc.collect()

print(f"REFERENCE_DATE: {REFERENCE_DATE}")
log_step('SETUP', f'sample_user_ids cargado: {N_SAMPLE:,} usuarios (hex limpio)')
log_step('SETUP', f'REFERENCE_DATE: {REFERENCE_DATE}')

## 1. Carga del CSV original y exploración

In [4]:
# [EXEC] Carga de currency_transactions.csv
t0 = time.time()

dtypes_in = {
    '_id': 'str', 'user_id': 'str', 'concept': 'int32',
    'currency': 'str', 'quantity': 'int64',
    'final_quantity': 'float64', 'updated_at': 'str', 'created_at': 'str',
}

df_raw = pd.read_csv(CSV_INPUT, dtype=dtypes_in, low_memory=False)
load_time = time.time() - t0

print(f"Shape: {df_raw.shape}")
print(f"Tiempo de carga: {load_time:.1f}s")
print(f"Memoria: {df_raw.memory_usage(deep=True).sum() / 1e9:.2f} GB")
log_step('EXEC', f'Carga CSV: shape={df_raw.shape}, tiempo={load_time:.1f}s')

Shape: (5121333, 8)
Tiempo de carga: 7.7s
Memoria: 0.83 GB
[EXEC] 23:20:11 — Carga CSV: shape=(5121333, 8), tiempo=7.7s


In [5]:
# [ANALYSIS] Tipos de datos del CSV crudo — inspección detallada
print("TIPOS DE DATOS DEL CSV CRUDO")
print("=" * 70)
for col in df_raw.columns:
    dtype = df_raw[col].dtype
    n_nulls = df_raw[col].isnull().sum()
    pct_nulls = n_nulls / len(df_raw) * 100
    n_unique = df_raw[col].nunique()
    example = df_raw[col].dropna().iloc[0] if n_nulls < len(df_raw) else 'TODO NULO'
    print(f"  {col:<20} dtype={str(dtype):<10} nulls={n_nulls:>10,} ({pct_nulls:5.1f}%)  "
          f"unique={n_unique:>8,}  ejemplo='{str(example)[:40]}'")

TIPOS DE DATOS DEL CSV CRUDO


  _id                  dtype=str        nulls=         0 (  0.0%)  unique=5,121,333  ejemplo='ObjectId(69a8b97c32912f1fb07df3d0)'
  user_id              dtype=str        nulls= 2,578,073 ( 50.3%)  unique=  67,085  ejemplo='699e4986a0761a3558561bb9'
  concept              dtype=int32      nulls=         0 (  0.0%)  unique=      48  ejemplo='52'
  currency             dtype=str        nulls=    73,307 (  1.4%)  unique=      29  ejemplo='mana'
  quantity             dtype=int64      nulls=         0 (  0.0%)  unique=   5,490  ejemplo='-60'


  final_quantity       dtype=float64    nulls=         0 (  0.0%)  unique= 170,301  ejemplo='68.0'


  updated_at           dtype=str        nulls=         0 (  0.0%)  unique=5,115,027  ejemplo='2026-03-04T23:00:12.557Z'


  created_at           dtype=str        nulls=         0 (  0.0%)  unique=5,115,027  ejemplo='2026-03-04T23:00:12.557Z'


In [6]:
# [ANALYSIS] Nulos por columna
nulls = df_raw.isnull().sum().to_frame(name='n_nulls')
nulls['pct_nulls'] = (nulls['n_nulls'] / len(df_raw) * 100).round(2)
nulls = nulls.sort_values('pct_nulls', ascending=False)
print(nulls)
nulls.to_csv(REPORT_DIR / 'nulos_por_columna_raw.csv')
log_step('ANALYSIS', 'Nulos por columna guardados')

                n_nulls  pct_nulls
user_id         2578073      50.34
currency          73307       1.43
_id                   0       0.00
concept               0       0.00
quantity              0       0.00
final_quantity        0       0.00
updated_at            0       0.00
created_at            0       0.00
[ANALYSIS] 23:20:13 — Nulos por columna guardados


In [7]:
# [ANALYSIS] Distribución de currency y concept en el CSV crudo COMPLETO
print("=" * 70)
print("CURRENCIES — DATASET COMPLETO (5.1M filas)")
print("=" * 70)
vc_curr = df_raw['currency'].value_counts(dropna=False)
for c, n in vc_curr.head(20).items():
    pct = n / len(df_raw) * 100
    print(f"  {str(c)[:30]:<30} {n:>12,} ({pct:5.2f}%)")
print(f"Total currencies distintas: {df_raw['currency'].nunique()}")

print("\n" + "=" * 70)
print("CONCEPTS — DATASET COMPLETO (top 20)")
print("=" * 70)
vc_conc = df_raw['concept'].value_counts().head(20)
for c, n in vc_conc.items():
    pct = n / len(df_raw) * 100
    print(f"  concept {str(c):<8} {n:>12,} ({pct:5.2f}%)")
print(f"Total concepts distintos: {df_raw['concept'].nunique()}")

vc_curr.to_csv(REPORT_DIR / 'distribucion_currency_raw.csv')
vc_conc.to_csv(REPORT_DIR / 'distribucion_concept_raw.csv')
log_step('ANALYSIS', 'Distribuciones currency/concept (crudo) guardadas')

CURRENCIES — DATASET COMPLETO (5.1M filas)
  gold                              1,253,563 (24.48%)
  pve_energy                        1,210,115 (23.63%)
  experience                        1,015,347 (19.83%)
  dark_steel                          963,742 (18.82%)
  mana                                253,657 ( 4.95%)
  gems                                166,926 ( 3.26%)
  tower_event_points                   98,507 ( 1.92%)
  nan                                  73,307 ( 1.43%)
  energy_pack_1                        20,855 ( 0.41%)
  protection_pack_1                    16,991 ( 0.33%)
  protection_pack_2                    16,455 ( 0.32%)
  energy_pack_2                        11,585 ( 0.23%)
  mana_pack_1                           3,633 ( 0.07%)
  mana_pack_2                           3,195 ( 0.06%)
  energy_pack_3                         2,854 ( 0.06%)
  item_level_up                         2,660 ( 0.05%)
  runes                                 2,372 ( 0.05%)
  energy_pack_4       

In [8]:
# [ANALYSIS] Desglose de monedas: transacciones de USUARIO vs SISTEMA
has_uid = df_raw['user_id'].notna()

print("=" * 70)
print("CURRENCIES en filas CON user_id (datos de usuario)")
print("=" * 70)
dist_user = df_raw[has_uid]['currency'].value_counts(dropna=False).head(15)
for c, n in dist_user.items():
    pct = n / has_uid.sum() * 100
    print(f"  {str(c)[:30]:<30} {n:>12,} ({pct:5.2f}%)")

print("\n" + "=" * 70)
print("CURRENCIES en filas SIN user_id (transacciones del sistema)")
print("=" * 70)
dist_sys = df_raw[~has_uid]['currency'].value_counts(dropna=False).head(10)
for c, n in dist_sys.items():
    pct = n / (~has_uid).sum() * 100
    print(f"  {str(c)[:30]:<30} {n:>12,} ({pct:5.2f}%)")

# Confirmar monedas del sistema
system_currencies = ['pve_energy', 'experience', 'mana', 'tower_event_points']
print("\n" + "=" * 70)
print("VERIFICACIÓN: monedas del sistema con user_id")
print("=" * 70)
for cur in system_currencies:
    count = ((df_raw['currency'] == cur) & has_uid).sum()
    status = "0 (confirmado sistema)" if count == 0 else f"{count:,} (tiene datos de usuario!)"
    print(f"  {cur:<25} {status}")

log_step('ANALYSIS', 'Desglose monedas usuario vs sistema completado')

CURRENCIES en filas CON user_id (datos de usuario)


  gold                              1,253,563 (49.29%)
  dark_steel                          963,742 (37.89%)
  gems                                166,926 ( 6.56%)
  nan                                  73,307 ( 2.88%)
  energy_pack_1                        20,855 ( 0.82%)
  protection_pack_1                    16,991 ( 0.67%)
  protection_pack_2                    16,455 ( 0.65%)
  energy_pack_2                        11,585 ( 0.46%)
  mana_pack_1                           3,633 ( 0.14%)
  mana_pack_2                           3,195 ( 0.13%)
  energy_pack_3                         2,854 ( 0.11%)
  item_level_up                         2,660 ( 0.10%)
  runes                                 2,372 ( 0.09%)
  energy_pack_4                         1,477 ( 0.06%)
  mana_pack_3                           1,471 ( 0.06%)

CURRENCIES en filas SIN user_id (transacciones del sistema)


  pve_energy                        1,210,115 (46.94%)
  experience                        1,015,347 (39.38%)
  mana                                253,657 ( 9.84%)
  tower_event_points                   98,507 ( 3.82%)
  rage                                    447 ( 0.02%)

VERIFICACIÓN: monedas del sistema con user_id
  pve_energy                0 (confirmado sistema)
  experience                0 (confirmado sistema)
  mana                      0 (confirmado sistema)
  tower_event_points        0 (confirmado sistema)
[ANALYSIS] 23:20:14 — Desglose monedas usuario vs sistema completado


In [ ]:
# [ANALYSIS] Rango temporal
df_raw['created_dt'] = pd.to_datetime(df_raw['created_at'], errors='coerce', utc=True).dt.tz_localize(None)

# Filtro cutoff: descartar transacciones posteriores al cutoff
_cut_naive = CUTOFF_DATE.tz_localize(None) if CUTOFF_DATE.tz is not None else CUTOFF_DATE
n_pre_cut = len(df_raw)
df_raw = df_raw[df_raw['created_dt'].notna() & (df_raw['created_dt'] <= _cut_naive)].copy()
log_step('EXEC', f'Filtro cutoff (created_dt <= CUTOFF): {n_pre_cut:,} → {len(df_raw):,}')

print(f"Rango temporal: {df_raw['created_dt'].min()} → {df_raw['created_dt'].max()}")
print(f"Duración: {(df_raw['created_dt'].max() - df_raw['created_dt'].min()).days} días")

by_year = df_raw['created_dt'].dt.year.value_counts().sort_index()
print("\nTransacciones por año:")
for y, n in by_year.items():
    print(f"  {int(y) if pd.notna(y) else 'NaT':<6} {n:>12,}")

log_step('ANALYSIS', f'Rango temporal: {df_raw["created_dt"].min()} → {df_raw["created_dt"].max()}')

## 2. Filtrado por sample_user_ids

Aplicamos el principio de robustez: filtramos por `sample_user_ids.parquet`.

**Importante:** se filtra por la columna `user_id` (hex del usuario),
NO por `_id` (ObjectId de la transacción). La columna `_id` identifica
cada transacción individual, no al usuario.

In [10]:
# [EXEC] Filtrado por sample_user_ids
t0 = time.time()
n_before = len(df_raw)

# 1. Eliminar filas con user_id nulo (transacciones del sistema)
df_filtered = df_raw.dropna(subset=['user_id']).copy()
n_no_nulls = len(df_filtered)

# 2. Filtrar por sample_user_ids (ambos en formato hex limpio)
df_filtered = df_filtered[df_filtered['user_id'].isin(sample_user_ids)].copy()
n_after = len(df_filtered)

filter_time = time.time() - t0

# Usuarios únicos con transacciones
n_users_with_tx = df_filtered['user_id'].nunique()
n_users_without_tx = N_SAMPLE - n_users_with_tx

print(f"Filas iniciales (CSV crudo):      {n_before:>12,}")
print(f"Tras eliminar user_id nulo:       {n_no_nulls:>12,}  ({n_no_nulls/n_before*100:.1f}%)")
print(f"Tras filtro sample_user_ids:      {n_after:>12,}  ({n_after/n_before*100:.1f}%)")
print(f"Tiempo de filtrado: {filter_time:.1f}s")
print(f"\nUsuarios CON transacciones:  {n_users_with_tx:>8,} ({n_users_with_tx/N_SAMPLE*100:.1f}%)")
print(f"Usuarios SIN transacciones:  {n_users_without_tx:>8,} ({n_users_without_tx/N_SAMPLE*100:.1f}%)")

log_step('EXEC', f'Filtrado: {n_before:,} → {n_after:,} ({n_after/n_before*100:.1f}%)')
log_step('EXEC', f'Usuarios con tx: {n_users_with_tx:,}, sin tx: {n_users_without_tx:,}')

# Liberar CSV crudo
del df_raw
gc.collect()

Filas iniciales (CSV crudo):         5,121,333
Tras eliminar user_id nulo:          2,543,260  (49.7%)
Tras filtro sample_user_ids:         2,388,517  (46.6%)
Tiempo de filtrado: 0.5s

Usuarios CON transacciones:    42,077 (33.3%)
Usuarios SIN transacciones:    84,176 (66.7%)
[EXEC] 23:20:16 — Filtrado: 5,121,333 → 2,388,517 (46.6%)
[EXEC] 23:20:16 — Usuarios con tx: 42,077, sin tx: 84,176


0

In [11]:
# [ANALYSIS] Distribución de currency y concept DESPUÉS del filtrado
print("=" * 70)
print("CURRENCIES — POST-FILTRO (solo transacciones de usuarios)")
print("=" * 70)
vc_post = df_filtered['currency'].value_counts(dropna=False).head(15)
for c, n in vc_post.items():
    pct = n / len(df_filtered) * 100
    print(f"  {str(c)[:30]:<30} {n:>12,} ({pct:5.2f}%)")

print("\n" + "=" * 70)
print("CONCEPTS — POST-FILTRO (top 15)")
print("=" * 70)
vc_conc_post = df_filtered['concept'].value_counts().head(15)
for c, n in vc_conc_post.items():
    pct = n / len(df_filtered) * 100
    print(f"  concept {str(c):<8} {n:>12,} ({pct:5.2f}%)")

vc_post.to_csv(REPORT_DIR / 'distribucion_currency_postfiltro.csv')
vc_conc_post.to_csv(REPORT_DIR / 'distribucion_concept_postfiltro.csv')
log_step('ANALYSIS', 'Distribuciones currency/concept post-filtro guardadas')

CURRENCIES — POST-FILTRO (solo transacciones de usuarios)
  gold                              1,159,563 (48.55%)
  dark_steel                          920,803 (38.55%)
  gems                                165,266 ( 6.92%)
  nan                                  58,317 ( 2.44%)
  energy_pack_1                        20,678 ( 0.87%)
  protection_pack_1                    16,514 ( 0.69%)
  protection_pack_2                    15,991 ( 0.67%)
  energy_pack_2                        11,567 ( 0.48%)
  mana_pack_1                           3,633 ( 0.15%)
  mana_pack_2                           3,190 ( 0.13%)
  energy_pack_3                         2,848 ( 0.12%)
  item_level_up                         2,657 ( 0.11%)
  runes                                 2,371 ( 0.10%)
  energy_pack_4                         1,476 ( 0.06%)
  mana_pack_3                           1,471 ( 0.06%)

CONCEPTS — POST-FILTRO (top 15)
  concept 4             809,872 (33.91%)
  concept 2             484,123 (20.27%)
  

In [12]:
# [ANALYSIS] Estadísticas post-filtro
print(f"Filas filtradas: {len(df_filtered):,}")
print(f"Usuarios únicos: {df_filtered['user_id'].nunique():,}")
print(f"Memoria: {df_filtered.memory_usage(deep=True).sum() / 1e6:.1f} MB")

# Transacciones por usuario
tx_per_user = df_filtered.groupby('user_id').size()
print(f"\nTransacciones por usuario:")
print(f"  Mediana: {tx_per_user.median():.0f}")
print(f"  Media:   {tx_per_user.mean():.1f}")
print(f"  Min:     {tx_per_user.min()}")
print(f"  Max:     {tx_per_user.max()}")
print(f"  P90:     {tx_per_user.quantile(0.90):.0f}")
print(f"  P99:     {tx_per_user.quantile(0.99):.0f}")

print(f"\nTipos de datos del DataFrame filtrado:")
for col in df_filtered.columns:
    print(f"  {col:<20} {str(df_filtered[col].dtype):<15}")

log_step('ANALYSIS', f'Stats post-filtro: {len(df_filtered):,} filas, {df_filtered["user_id"].nunique():,} usuarios')

Filas filtradas: 2,388,517
Usuarios únicos: 42,077
Memoria: 451.9 MB

Transacciones por usuario:
  Mediana: 14
  Media:   56.8
  Min:     1
  Max:     20561
  P90:     101
  P99:     678

Tipos de datos del DataFrame filtrado:
  _id                  str            
  user_id              str            
  concept              int32          
  currency             str            
  quantity             int64          
  final_quantity       float64        
  updated_at           str            
  created_at           str            
  created_dt           datetime64[us] 
[ANALYSIS] 23:20:16 — Stats post-filtro: 2,388,517 filas, 42,077 usuarios


In [13]:
# [EXEC] Guardar parquet intermedio (transacciones filtradas a nivel individual)
cols_to_keep = ['user_id', 'concept', 'currency', 'quantity', 'final_quantity', 'created_dt']
df_filtered[cols_to_keep].to_parquet(PARQUET_FILTERED, index=False, compression='snappy')

size_mb = PARQUET_FILTERED.stat().st_size / 1e6
log_step('EXEC', f'Parquet intermedio: {len(df_filtered):,} filas, {size_mb:.1f} MB')
print(f"Guardado: {PARQUET_FILTERED} ({size_mb:.1f} MB)")

[EXEC] 23:20:17 — Parquet intermedio: 2,388,517 filas, 32.0 MB
Guardado: /Users/jezquerro/Documents/tfg/data/data_qc/currency_transactions_filtered.parquet (32.0 MB)


## 3. Agregación por usuario — generación de features

Compactamos las transacciones individuales a **1 fila por usuario**,
generando features en 6 grupos:

- **Grupo A:** Volumen y diversidad (5 features)
- **Grupo B:** Actividad temporal real (6 features)
- **Grupo C:** Por moneda top — gold, gems, dark_steel (3 × 5 = 15 features)
- **Grupo D:** Otras monedas agrupadas (3 features)
- **Grupo E:** Por concept top — 7 concepts (7 features)
- **Grupo F:** Otros concepts agrupados (1 feature)

**Total: ~37 features + user_id = 38 columnas**

In [14]:
# [EXEC] Grupo A: Volumen y diversidad (5 features)
t0 = time.time()

group_a = df_filtered.groupby('user_id').agg(
    tx_count_total=('concept', 'size'),
    tx_unique_concepts=('concept', 'nunique'),
    tx_unique_currencies=('currency', 'nunique'),
    tx_first_date=('created_dt', 'min'),
    tx_last_date=('created_dt', 'max'),
)

print(f"Grupo A: {len(group_a):,} filas, {group_a.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo A (volumen): {group_a.shape[1]} features')

Grupo A: 42,077 filas, 5 features, 0.1s
[EXEC] 23:20:17 — Grupo A (volumen): 5 features


In [ ]:
# [EXEC] Grupo B: Actividad temporal real (6 features)
t0 = time.time()

df_filtered['date_only'] = df_filtered['created_dt'].dt.date
days_active = df_filtered.groupby('user_id')['date_only'].nunique().rename('tx_days_active')

cutoff_7d  = CUTOFF_DATE - pd.Timedelta(days=7)
cutoff_14d = CUTOFF_DATE - pd.Timedelta(days=14)
cutoff_30d = CUTOFF_DATE - pd.Timedelta(days=30)

tx_last_7d  = df_filtered[df_filtered['created_dt'] >= cutoff_7d ].groupby('user_id').size().rename('tx_count_last_7d')
tx_last_14d = df_filtered[df_filtered['created_dt'] >= cutoff_14d].groupby('user_id').size().rename('tx_count_last_14d')
tx_last_30d = df_filtered[df_filtered['created_dt'] >= cutoff_30d].groupby('user_id').size().rename('tx_count_last_30d')

group_b = pd.concat([days_active, tx_last_7d, tx_last_14d, tx_last_30d], axis=1)
group_b = group_b.fillna(0).astype('int32')

group_b['tx_per_active_day'] = (
    group_a['tx_count_total'] / group_b['tx_days_active'].replace(0, np.nan)
).fillna(0).round(2)

expected_weekly = (group_a['tx_count_total'] / group_b['tx_days_active'].replace(0, np.nan)) * 7
group_b['tx_recency_decay'] = (group_b['tx_count_last_7d'] / expected_weekly).fillna(0).round(2)

df_filtered.drop(columns=['date_only'], inplace=True)

print(f"Grupo B: {len(group_b):,} filas, {group_b.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo B (temporal): {group_b.shape[1]} features')

In [16]:
# [EXEC] Grupo C: Features por moneda top (3 monedas × 5 = 15 features)
t0 = time.time()

group_c_dfs = []

for currency in TOP_CURRENCIES:
    sub = df_filtered[df_filtered['currency'] == currency]

    if len(sub) == 0:
        print(f"  {currency}: 0 transacciones (esto no debería pasar)")
        continue

    earned = sub[sub['quantity'] > 0].groupby('user_id')['quantity'].sum().rename(f'{currency}_total_earned')
    spent = sub[sub['quantity'] < 0].groupby('user_id')['quantity'].sum().abs().rename(f'{currency}_total_spent')
    tx_count = sub.groupby('user_id').size().rename(f'{currency}_tx_count')
    last_balance = (
        sub.sort_values('created_dt')
           .groupby('user_id')['final_quantity']
           .last()
           .rename(f'{currency}_last_balance')
    )

    curr_features = pd.concat([earned, spent, tx_count, last_balance], axis=1).fillna(0)
    curr_features[f'{currency}_net_balance'] = (
        curr_features[f'{currency}_total_earned'] - curr_features[f'{currency}_total_spent']
    )

    curr_features = curr_features[[
        f'{currency}_total_earned', f'{currency}_total_spent',
        f'{currency}_net_balance', f'{currency}_tx_count', f'{currency}_last_balance',
    ]]

    group_c_dfs.append(curr_features)
    print(f"  {currency}: {len(curr_features):,} usuarios con actividad")

group_c = pd.concat(group_c_dfs, axis=1)
print(f"\nGrupo C: {len(group_c):,} filas, {group_c.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo C (por moneda top): {group_c.shape[1]} features')

  gold: 41,012 usuarios con actividad
  gems: 13,937 usuarios con actividad


  dark_steel: 37,998 usuarios con actividad

Grupo C: 41,639 filas, 15 features, 0.5s
[EXEC] 23:20:18 — Grupo C (por moneda top): 15 features


In [17]:
# [EXEC] Grupo D: Otras monedas agrupadas (3 features)
t0 = time.time()

mask_other = ~df_filtered['currency'].isin(TOP_CURRENCIES) & df_filtered['currency'].notna()
df_other = df_filtered[mask_other]

if len(df_other) > 0:
    group_d = df_other.groupby('user_id').agg(
        other_currency_tx_count=('currency', 'size'),
        other_currency_total_movement=('quantity', lambda x: x.abs().sum()),
        other_currency_unique=('currency', 'nunique'),
    )
else:
    group_d = pd.DataFrame(columns=['other_currency_tx_count', 'other_currency_total_movement', 'other_currency_unique'])

print(f"Grupo D: {len(group_d):,} filas, {group_d.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo D (otras monedas): {group_d.shape[1]} features')

Grupo D: 16,463 filas, 3 features, 0.2s
[EXEC] 23:20:18 — Grupo D (otras monedas): 3 features


In [18]:
# [EXEC] Grupo E: Features por concept top (7 features)
t0 = time.time()

concept_counts = (
    df_filtered[df_filtered['concept'].isin(TOP_CONCEPTS)]
    .groupby(['user_id', 'concept'])
    .size()
    .unstack(fill_value=0)
)

concept_counts.columns = [f'concept_{int(c)}_count' for c in concept_counts.columns]

for c in TOP_CONCEPTS:
    col_name = f'concept_{c}_count'
    if col_name not in concept_counts.columns:
        concept_counts[col_name] = 0

concept_counts = concept_counts[[f'concept_{c}_count' for c in TOP_CONCEPTS]].astype('int32')
group_e = concept_counts

print(f"Grupo E: {len(group_e):,} filas, {group_e.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo E (por concept top): {group_e.shape[1]} features')

Grupo E: 41,969 filas, 7 features, 0.2s
[EXEC] 23:20:18 — Grupo E (por concept top): 7 features


In [19]:
# [EXEC] Grupo F: Otros concepts agrupados (1 feature)
t0 = time.time()

group_f = (
    df_filtered[~df_filtered['concept'].isin(TOP_CONCEPTS)]
    .groupby('user_id')
    .size()
    .rename('other_concepts_count')
    .to_frame()
    .astype('int32')
)

print(f"Grupo F: {len(group_f):,} filas, {group_f.shape[1]} feature, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo F (otros concepts): 1 feature')

Grupo F: 17,758 filas, 1 feature, 0.0s
[EXEC] 23:20:18 — Grupo F (otros concepts): 1 feature


In [20]:
# [EXEC] Combinar todos los grupos + reindex con TODOS los user_ids del sample
t0 = time.time()

features = pd.concat([group_a, group_b, group_c, group_d, group_e, group_f], axis=1)

# CRÍTICO: reindex con TODOS los user_ids del sample
features = features.reindex(list(sample_user_ids))

# Rellenar nulos con 0 para columnas numéricas (excepto fechas)
date_cols = ['tx_first_date', 'tx_last_date']
numeric_cols = [c for c in features.columns if c not in date_cols]
features[numeric_cols] = features[numeric_cols].fillna(0)

# Convertir índice a columna
features = features.reset_index().rename(columns={'index': 'user_id'})

# Verificación crítica
assert len(features) == N_SAMPLE, \
    f"ERROR: features tiene {len(features)} filas, sample tiene {N_SAMPLE}"

print(f"Features finales: {features.shape}")
print(f"Verificación: {len(features)} filas == {N_SAMPLE} usuarios del sample")
print(f"Tiempo combinación: {time.time()-t0:.1f}s")
log_step('EXEC', f'Features combinadas: {features.shape[0]:,} × {features.shape[1]} cols')

Features finales: (126253, 38)
Verificación: 126253 filas == 126253 usuarios del sample
Tiempo combinación: 0.1s
[EXEC] 23:20:18 — Features combinadas: 126,253 × 38 cols


## 4. Validación de features generadas

In [21]:
# [ANALYSIS] Tipos de datos de las features finales
print("TIPOS DE DATOS — FEATURES FINALES")
print("=" * 70)
for col in features.columns:
    dtype = features[col].dtype
    n_nulls = features[col].isnull().sum()
    n_zeros = (features[col] == 0).sum() if dtype != 'object' and str(dtype) != 'datetime64[ns]' else 'N/A'
    n_nonzero = len(features) - n_nulls - (n_zeros if isinstance(n_zeros, (int, np.integer)) else 0)
    print(f"  {col:<30} dtype={str(dtype):<15} nulls={n_nulls:>8,}  "
          f"zeros={str(n_zeros):>8}  nonzero={str(n_nonzero):>8}")

TIPOS DE DATOS — FEATURES FINALES
  user_id                        dtype=str             nulls=       0  zeros=       0  nonzero=  126253
  tx_count_total                 dtype=float64         nulls=       0  zeros=   84176  nonzero=   42077
  tx_unique_concepts             dtype=float64         nulls=       0  zeros=   84176  nonzero=   42077
  tx_unique_currencies           dtype=float64         nulls=       0  zeros=   84187  nonzero=   42066
  tx_first_date                  dtype=datetime64[us]  nulls=  84,176  zeros=       0  nonzero=   42077
  tx_last_date                   dtype=datetime64[us]  nulls=  84,176  zeros=       0  nonzero=   42077
  tx_days_active                 dtype=float64         nulls=       0  zeros=   84176  nonzero=   42077
  tx_count_last_7d               dtype=float64         nulls=       0  zeros=  115127  nonzero=   11126
  tx_count_last_14d              dtype=float64         nulls=       0  zeros=  105922  nonzero=   20331
  tx_count_last_30d           

In [22]:
# [ANALYSIS] Investigación de nulos en features finales
nulls_f = features.isnull().sum().to_frame(name='n_nulls')
nulls_f['pct_nulls'] = (nulls_f['n_nulls'] / len(features) * 100).round(2)
nulls_f = nulls_f[nulls_f['n_nulls'] > 0].sort_values('pct_nulls', ascending=False)

if len(nulls_f) > 0:
    print("Columnas con nulos (esperado solo en fechas):")
    print(nulls_f)
else:
    print("No hay nulos en features finales")

nulls_f.to_csv(REPORT_DIR / 'nulos_features_final.csv')
log_step('ANALYSIS', f'Nulos en features: {len(nulls_f)} columnas con nulos')

Columnas con nulos (esperado solo en fechas):
               n_nulls  pct_nulls
tx_first_date    84176      66.67
tx_last_date     84176      66.67
[ANALYSIS] 23:20:18 — Nulos en features: 2 columnas con nulos


In [23]:
# [ANALYSIS] Estadísticas descriptivas
numeric_features = features.select_dtypes(include=[np.number])
desc = numeric_features.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).T.round(2)
print(desc)
desc.to_csv(REPORT_DIR / 'features_describe.csv')
log_step('ANALYSIS', 'Estadísticas descriptivas guardadas')

                                  count      mean         std        min  25%  50%     75%     90%       99%           max
tx_count_total                 126253.0     18.92      172.52        0.0  0.0  0.0     7.0    31.0    265.48  2.056100e+04
tx_unique_concepts             126253.0      1.91        3.13        0.0  0.0  0.0     4.0     6.0     12.00  2.700000e+01
tx_unique_currencies           126253.0      1.05        1.82        0.0  0.0  0.0     2.0     4.0      7.00  1.800000e+01
tx_days_active                 126253.0      0.70        1.94        0.0  0.0  0.0     1.0     2.0      8.00  3.200000e+01
tx_count_last_7d               126253.0      4.05       45.21        0.0  0.0  0.0     0.0     0.0     87.00  5.182000e+03
tx_count_last_14d              126253.0      8.34       83.59        0.0  0.0  0.0     0.0     9.0    158.00  9.996000e+03
tx_count_last_30d              126253.0     18.43      167.98        0.0  0.0  0.0     7.0    30.0    262.00  1.961700e+04
tx_per_active_da

In [24]:
# [ANALYSIS] Histogramas de features clave (solo las que tienen datos)
key_features = [
    'tx_count_total', 'tx_days_active', 'tx_count_last_7d',
    'gold_total_earned', 'gems_total_earned', 'dark_steel_total_earned',
    'gold_tx_count', 'gems_tx_count', 'dark_steel_tx_count',
]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, feat in zip(axes.flatten(), key_features):
    if feat in features.columns:
        data = features[feat][features[feat] > 0]
        if len(data) > 0:
            ax.hist(np.log1p(data), bins=50, color='steelblue', alpha=0.7)
            ax.set_title(f'{feat} (log1p, n={len(data):,})')
            ax.set_ylabel('count')
        else:
            ax.set_title(f'{feat} (sin datos)')
    else:
        ax.set_title(f'{feat} (no existe)')

plt.tight_layout()
plt.savefig(REPORT_DIR / 'features_distributions.png', dpi=100, bbox_inches='tight')
plt.close()
log_step('ANALYSIS', 'Histogramas guardados')

[ANALYSIS] 23:20:19 — Histogramas guardados


In [ ]:
# [EXEC] Guardar features_currency_cutoff.parquet
features.to_parquet(PARQUET_FEATURES, index=False, compression='snappy')
size_mb = PARQUET_FEATURES.stat().st_size / 1e6
log_step('EXEC', f'features_currency_cutoff.parquet: {features.shape}, {size_mb:.1f} MB')
print(f"Guardado: {PARQUET_FEATURES} ({size_mb:.1f} MB)")

In [26]:
# [EXEC] Guardar muestra y liberar memoria
features.head(20).to_csv(REPORT_DIR / 'sample_head.csv', index=False)
log_step('EXEC', 'sample_head.csv guardado')

del df_filtered
gc.collect()
print("Memoria liberada")

[EXEC] 23:20:19 — sample_head.csv guardado
Memoria liberada


## 5. Informe de ejecución

In [ ]:
# [REPORT] Generar execution_report.md
t_total = time.time() - NOTEBOOK_START
t_min = int(t_total // 60)
t_sec = int(t_total % 60)
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

n_con_tx = features[features['tx_count_total'] > 0].shape[0]
n_sin_tx = features[features['tx_count_total'] == 0].shape[0]

lines = []

lines += [
    "# Informe de ejecucion — currency_transactions.csv",
    "",
    f"**Notebook:** notebooks/fase1_cleaning/02b_currency_transactions.ipynb",
    f"**Fecha:** {now_str}",
    f"**Tiempo de ejecucion:** {t_min} min {t_sec}s",
    f"**CSV de entrada:** data/data_raw/currency_transactions.csv (573 MB, {n_before:,} filas)",
    f"**Parquet filtrado:** data/data_qc/currency_transactions_filtered.parquet ({n_after:,} filas)",
    f"**Parquet features:** data/data_qc/features_currency_cutoff.parquet ({features.shape[0]:,} filas × {features.shape[1]} cols)",
    "",
    "---", "",
    "## Advertencia: monedas del sistema",
    "",
    "Las monedas `pve_energy`, `experience`, `mana` y `tower_event_points` son",
    "**100% transacciones del sistema** (user_id siempre nulo). Al filtrar por",
    "usuarios, TODAS sus transacciones desaparecen. Por eso NO se generan features",
    "para estas monedas — serían columnas de ceros puros.",
    "",
    "---", "",
    "## Resumen ejecutivo",
    "",
    f"Se procesaron {n_before:,} transacciones de `currency_transactions.csv`. Tras eliminar",
    f"transacciones del sistema (~50%, user_id nulo) y filtrar por los {N_SAMPLE:,} usuarios",
    f"del sample, quedaron {n_after:,} transacciones ({n_after/n_before*100:.1f}%).",
    "",
    f"Se generaron {features.shape[1] - 1} features en 6 grupos. Solo se usaron 3 monedas",
    f"(gold, gems, dark_steel) y 7 concepts con datos reales a nivel de usuario.",
    "",
    f"- Usuarios con transacciones: {n_con_tx:,} ({n_con_tx/len(features)*100:.1f}%)",
    f"- Usuarios sin transacciones: {n_sin_tx:,} ({n_sin_tx/len(features)*100:.1f}%)",
    "",
    "---", "",
    "## Constantes utilizadas",
    "",
    "| Constante | Valor |",
    "|---|---|",
    f"| `TOP_CURRENCIES` | {TOP_CURRENCIES} |",
    f"| `TOP_CONCEPTS` | {TOP_CONCEPTS} |",
    f"| `REFERENCE_DATE` | {REFERENCE_DATE} |",
    f"| `N_SAMPLE` | {N_SAMPLE:,} |",
    "",
    "---", "",
    "## Pasos ejecutados",
    "",
]
for s in steps_log:
    lines.append(f"- {s}")

lines += [
    "",
    "---", "",
    "## Filtrado aplicado",
    "",
    "| Paso | Filas | % original |",
    "|---|---|---|",
    f"| CSV original | {n_before:,} | 100% |",
    f"| Eliminar user_id nulo | {n_no_nulls:,} | {n_no_nulls/n_before*100:.1f}% |",
    f"| Filtrar por sample_user_ids | {n_after:,} | {n_after/n_before*100:.1f}% |",
    "",
    "---", "",
    "## Features generadas ({features.shape[1] - 1} features + user_id)",
    "",
    "### Grupo A — Volumen y diversidad (5)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `tx_count_total` | Transacciones totales |",
    "| `tx_unique_concepts` | Concepts distintos |",
    "| `tx_unique_currencies` | Monedas distintas |",
    "| `tx_first_date` | Primera transacción |",
    "| `tx_last_date` | Última transacción |",
    "",
    "### Grupo B — Temporal (6)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `tx_days_active` | Días con actividad |",
    "| `tx_count_last_7d` | Tx últimos 7 días |",
    "| `tx_count_last_14d` | Tx últimos 14 días |",
    "| `tx_count_last_30d` | Tx últimos 30 días |",
    "| `tx_per_active_day` | Tx por día activo |",
    "| `tx_recency_decay` | Ratio actividad reciente vs histórica |",
    "",
    "### Grupo C — Por moneda (3 × 5 = 15)",
    "Para gold, gems, dark_steel:",
    "| Feature | Descripción |",
    "|---|---|",
    "| `{currency}_total_earned` | Total ganado |",
    "| `{currency}_total_spent` | Total gastado |",
    "| `{currency}_net_balance` | earned - spent |",
    "| `{currency}_tx_count` | N transacciones |",
    "| `{currency}_last_balance` | Saldo final |",
    "",
    "### Grupo D — Otras monedas (3)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `other_currency_tx_count` | Tx con monedas no-top |",
    "| `other_currency_total_movement` | Movimiento absoluto |",
    "| `other_currency_unique` | Monedas distintas |",
    "",
    "### Grupo E — Por concept (7)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `concept_4_count` | gold — recompensa victoria |",
    "| `concept_2_count` | gold+dark_steel — equipamiento |",
    "| `concept_0_count` | dark_steel — recompensa |",
    "| `concept_10_count` | dark_steel+gems — mejora premium |",
    "| `concept_12_count` | gold — gasto menor |",
    "| `concept_46_count` | currency nula — investigar |",
    "| `concept_54_count` | gold — otro flujo |",
    "",
    "### Grupo F — Otros concepts (1)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `other_concepts_count` | Tx con concepts no-top |",
    "",
    "---", "",
    "## Monedas del juego",
    "",
    "| Moneda | Tipo | user_id | Fuente |",
    "|---|---|---|---|",
    "| gold | Básica | Datos usuario | Confirmada |",
    "| gems | Premium | Datos usuario | Confirmada |",
    "| dark_steel | Material | Datos usuario | Confirmada |",
    "| pve_energy | Energía | Solo sistema | Verificado |",
    "| experience | Progresión | Solo sistema | Verificado |",
    "| mana | Habilidades | Solo sistema | Verificado |",
    "| tower_event_points | Eventos | Solo sistema | Verificado |",
    "",
    "---", "",
    "## Lo que ha ido bien",
    "",
    "- Filtrado correcto por user_id (hex limpio, no _id)",
    "- Detección de monedas del sistema (100% user_id nulo)",
    "- Reducción de TOP_CURRENCIES de 6 a 3 (evita features vacías)",
    "- Reducción de TOP_CONCEPTS de 14 a 7",
    "- assert verifica que features tiene N_SAMPLE filas",
    "",
    "---", "",
    "## Lo que ha ido mal o requirió ajustes",
    "",
    "**Iteración 1:** user_id venía con formato ObjectId() → 0 matches en el filtro.",
    "  - **Causa:** regex incorrecta en 02a (buscaba comillas que no existen).",
    "  - **Solución:** corregida regex + función clean_user_id() como salvaguarda.",
    "",
    "**Iteración 1:** Features de pve_energy, experience, mana eran columnas de ceros.",
    "  - **Causa:** son transacciones del sistema (user_id siempre nulo).",
    "  - **Solución:** eliminadas del TOP_CURRENCIES en iteración 2.",
    "",
    "---", "",
    "## Decisiones tomadas",
    "",
    f"- REFERENCE_DATE heredado: {REFERENCE_DATE}",
    "- TOP_CURRENCIES reducido de 6 a 3 (eliminadas monedas del sistema)",
    "- TOP_CONCEPTS reducido de 14 a 7 (eliminados concepts del sistema)",
    "- Usuarios sin transacciones mantienen fila con valores 0",
    "",
    "---", "",
]

pf_size = PARQUET_FILTERED.stat().st_size / 1e6
feat_size = PARQUET_FEATURES.stat().st_size / 1e6
lines += [
    "## Archivos generados",
    "",
    "| Archivo | Tamaño |",
    "|---|---|",
    f"| currency_transactions_filtered.parquet | {pf_size:.1f} MB |",
    f"| features_currency_cutoff.parquet ({features.shape[1]} cols) | {feat_size:.1f} MB |",
    "| nulos_por_columna_raw.csv | - |",
    "| nulos_features_final.csv | - |",
    "| features_describe.csv | - |",
    "| features_distributions.png | - |",
    "| distribucion_currency_raw/postfiltro.csv | - |",
    "| distribucion_concept_raw/postfiltro.csv | - |",
    "| sample_head.csv | - |",
    "| execution_report.md | - |",
    "",
    "---", "",
    "## Advertencias para notebooks siguientes",
    "",
    "- Usar `user_id` (hex 24 chars) para joins, NUNCA `_id`",
    f"- REFERENCE_DATE = {REFERENCE_DATE}",
    "- ~65% de usuarios no tienen transacciones (features a cero)",
    "- `gems_total_earned > 0` confirma payer (compró con dinero real)",
    "- El mapeo de concepts es PROVISIONAL",
    "",
    "---", "",
    "## Tareas pendientes",
    "",
    "- [ ] Confirmar mapeo de concepts con la empresa",
    "- [ ] Investigar concept 46 (currency nula)",
    "- [ ] Feature derivada: `is_payer_tx = (gems_total_earned > 0)`",
    "- [ ] Feature derivada: `gold_spending_ratio`",
    "- [ ] Considerar spread temporal (std días entre tx)",
    "",
    "---", "",
    "## Diagrama del pipeline",
    "",
    "```",
    f"currency_transactions.csv ({n_before:,} filas)",
    "    │",
    f"    ├─ Eliminar user_id nulo    (-{n_before - n_no_nulls:,})",
    f"    ├─ Filtrar por sample        (-{n_no_nulls - n_after:,})",
    "    ▼",
    f"Filtrado ({n_after:,} filas) → currency_transactions_filtered.parquet",
    "    │",
    "    ├─ Grupo A: volumen (5)",
    "    ├─ Grupo B: temporal (6)",
    "    ├─ Grupo C: por moneda (15)",
    "    ├─ Grupo D: otras monedas (3)",
    "    ├─ Grupo E: por concept (7)",
    "    ├─ Grupo F: otros concepts (1)",
    "    ▼",
    f"features_currency_cutoff.parquet ({features.shape[0]:,} × {features.shape[1]})",
    "```",
    "",
    "---", "",
    "## Reproducibilidad",
    "",
    "1. Ejecutar 02a_users.ipynb primero",
    "2. Ejecutar 02b_currency_transactions.ipynb",
    f"3. Verificar: features_currency_cutoff.parquet == {N_SAMPLE:,} filas",
    "",
]

report_path = REPORT_DIR / 'execution_report.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f"Informe guardado: {report_path}")
log_step('REPORT', 'execution_report.md generado')

In [ ]:
# [REPORT] Resumen final
elapsed = time.time() - NOTEBOOK_START

print("=" * 70)
print("RESUMEN FINAL — Notebook 02b_currency_transactions")
print("=" * 70)
print(f"  Tiempo total          : {int(elapsed//60)}m {int(elapsed%60)}s")
print(f"  Filas CSV original    : {n_before:,}")
print(f"  Filas filtradas       : {n_after:,} ({n_after/n_before*100:.1f}%)")
print(f"  Usuarios con tx       : {n_users_with_tx:,} ({n_users_with_tx/N_SAMPLE*100:.1f}%)")
print(f"  Usuarios sin tx       : {n_users_without_tx:,} ({n_users_without_tx/N_SAMPLE*100:.1f}%)")
print(f"  Filas features final  : {len(features):,} (== {N_SAMPLE:,} sample)")
print(f"  Columnas features     : {features.shape[1]}")
print(f"  TOP_CURRENCIES        : {TOP_CURRENCIES}")
print(f"  TOP_CONCEPTS          : {TOP_CONCEPTS}")
print(f"  Monedas del sistema   : pve_energy, experience, mana, tower_event_points")
print()
print("Archivos generados:")
print(f"  currency_transactions_filtered.parquet ({PARQUET_FILTERED.stat().st_size/1e6:.1f} MB)")
print(f"  features_currency_cutoff.parquet ({PARQUET_FEATURES.stat().st_size/1e6:.1f} MB)")
print(f"  execution_report.md")
print(f"  Gráficos y CSVs en {REPORT_DIR}")
print("=" * 70)
print("Siguiente paso: revisar execution_report.md y compartirlo con Claude")

In [ ]:
# [REPORT] Logging dual: HTML + sección enriquecida del report
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from notebook_logging_template import (
    export_notebook_to_html, build_enriched_report_section,
)

notebook_path = PROJECT_ROOT / 'notebooks' / 'fase1_cleaning' / '02b_currency_transactions.ipynb'
html_path = REPORT_DIR / '02b_currency_transactions_run.html'
export_notebook_to_html(notebook_path, html_path)

enriched = build_enriched_report_section(
    df_final=features,
    raw_shape=(5121333, 8),
)
with open(REPORT_DIR / 'execution_report.md', 'a', encoding='utf-8') as f:
    f.write('\n\n---\n\n' + enriched)
print(f"Enriquecimiento appendeado a {REPORT_DIR / 'execution_report.md'}")